In [0]:
%run ../Includes/Copy-Datasets

In [0]:
%sql 
DESCRIBE TABLE customers_silver

col_name,data_type,comment
customer_id,string,null
email,string,null
first_name,string,null
last_name,string,null
gender,string,null
street,string,null
city,string,null
country,string,null
row_time,timestamp,null


We want to redact the email, first name, last name, and street fields from the data

In [0]:
%sql
CREATE OR REPLACE VIEW customers_vw AS
  SELECT
    customer_id,
    CASE 
      WHEN is_member('admins_demo') THEN email
      ELSE 'REDACTED'
    END AS email,
    gender,
    CASE 
      WHEN is_member('admins_demo') THEN first_name
      ELSE 'REDACTED'
    END AS first_name,
    CASE 
      WHEN is_member('admins_demo') THEN last_name
      ELSE 'REDACTED'
    END AS last_name,
    CASE 
      WHEN is_member('admins_demo') THEN street
      ELSE 'REDACTED'
    END AS street,
    city,
    country,
    row_time
  FROM customers_silver

We are currently not a member of the admins_demo group, so if we run a query to look at customers_silver table, we will not be able to see the information and the entire column will be REDACTED

In [0]:
%sql
SELECT * FROM customers_vw

customer_id,email,gender,first_name,last_name,street,city,country,row_time
C01154,REDACTED,Male,REDACTED,REDACTED,REDACTED,Mari,Brazil,2022-08-17T02:06:00Z
C01153,REDACTED,Female,REDACTED,REDACTED,REDACTED,Okinawa,Japan,2022-08-17T01:07:00Z
C01151,REDACTED,Female,REDACTED,REDACTED,REDACTED,Baklashi,Russia,2022-08-16T23:04:00Z
C01150,REDACTED,Male,REDACTED,REDACTED,REDACTED,Oshawa,Canada,2022-08-16T22:01:00Z
C01148,REDACTED,Female,REDACTED,REDACTED,REDACTED,Bantuanon,Philippines,2022-08-15T20:02:00Z
C01147,REDACTED,Female,REDACTED,REDACTED,REDACTED,Karolino-Buhaz,Ukraine,2022-08-15T19:06:00Z
C01145,REDACTED,Female,REDACTED,REDACTED,REDACTED,Amānzī,Afghanistan,2022-08-15T17:09:00Z
C01144,REDACTED,Female,REDACTED,REDACTED,REDACTED,Shiyan,China,2022-08-14T16:02:00Z
C01142,REDACTED,Non-binary,REDACTED,REDACTED,REDACTED,Nyachera,Uganda,2022-08-14T14:02:00Z
C01141,REDACTED,Female,REDACTED,REDACTED,REDACTED,Pukkila,Finland,2022-08-14T13:07:00Z


For row level access control 

In [0]:
%sql
CREATE OR REPLACE VIEW customers_fr_vw AS
SELECT * FROM customers_vw
WHERE 
  CASE 
    WHEN is_member('admins_demo') THEN TRUE
    ELSE country = "France" AND row_time > "2022-01-01"
  END

when we query, we should only be allowed to see records that have country = FRANCE and after the specified row_time date - the column information from the previous view are also still REDACTED since we are stacking this row level control dynamic view over the colum level control dynamic view (called customers_view)

In [0]:
%sql
SELECT * FROM customers_fr_vw

customer_id,email,gender,first_name,last_name,street,city,country,row_time
C01040,REDACTED,Female,REDACTED,REDACTED,REDACTED,Castelnaudary,France,2022-07-21T08:10:00Z
C01024,REDACTED,Agender,REDACTED,REDACTED,REDACTED,Castelsarrasin,France,2022-07-16T16:08:00Z
C01021,REDACTED,Male,REDACTED,REDACTED,REDACTED,Bordeaux,France,2022-07-16T13:02:00Z
C00966,REDACTED,Female,REDACTED,REDACTED,REDACTED,Marseille,France,2022-07-05T06:05:00Z
C00958,REDACTED,Female,REDACTED,REDACTED,REDACTED,Wissous,France,2022-06-30T22:05:00Z
C00924,REDACTED,Genderfluid,REDACTED,REDACTED,REDACTED,Marly,France,2022-06-23T12:09:00Z
C00922,REDACTED,Male,REDACTED,REDACTED,REDACTED,Saint-Avertin,France,2022-06-22T10:06:00Z
C00839,REDACTED,Female,REDACTED,REDACTED,REDACTED,Charenton-le-Pont,France,2022-06-01T23:04:00Z
C00835,REDACTED,Female,REDACTED,REDACTED,REDACTED,Abbeville,France,2022-06-01T19:06:00Z
C00697,REDACTED,Male,REDACTED,REDACTED,REDACTED,Castelsarrasin,France,2022-04-29T01:01:00Z


We can rerun this notebook after putting ourselves into the admin_demos group in the security groups settings